<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h1 align="center">Madaline: Separating Classes with Many Adalines</h1>

**Madaline** (*Many ADAptive LINEs*) was introduced by *Bernard Widrow* in the early **1960s**.  
A single Adaline or perceptron can only draw **one straight line**. Madaline puts several Adalines side by side in a **hidden layer** and combines their answers with a **fixed logic unit**, so it can separate classes that **no single line** can separate.

<br>

<div align="center">
    <img src="images/madaline.png" width="800">
    <br>
    <em><b>Madaline: hidden Adalines (trained) and a fixed output unit</b></em>
</div>

<br>

- Every hidden unit computes **net = w₁x₁ + w₂x₂ + b** and outputs **z = sign(net) ∈ {−1, +1}**, which is one **half-plane**.
- The **output unit has fixed weights** and does not learn. Here it is an **AND** gate: the output is **+1** only if **all** hidden units say **+1**.
- So the network answers **+1** inside the **intersection of the half-planes**, a convex region, and **−1** everywhere else.
- Training uses **Madaline Rule I (MR-I)**: when the network is wrong, only the hidden unit(s) that can fix the answer are corrected.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">The Data</h2>

Two small classes of 2-D points:

- **Class 1** (orange, target **+1**): 6 points in the **middle**
- **Class 2** (blue, target **−1**): 6 points that **surround** them (left, right, top and bottom)

<div align="center">
    <img src="images/data_b.png" width="550">
</div>

The orange points are **inside** the blue ones, so the two classes are **not linearly separable**.

</div>

In [1]:
# The two classes (x1, x2)
CLASS1 = [(0.5, 2.5), (1, 2), (1, 3), (1.25, 2.5), (1.5, 2), (1.5, 3)]   # orange, target +1
CLASS2 = [(0, 2), (0, 3), (1, 1), (1, 4), (2, 2), (2, 4)]                # blue,   target -1

DATA_B = [(p, 1) for p in CLASS1] + [(p, -1) for p in CLASS2]

if __name__ == "__main__":
    print("Class   |  x1    x2  | target")
    print("----------------------------")

    for (x1, x2), t in DATA_B:
        print(f"Class {1 if t == 1 else 2} | {x1:4.2f}  {x2:4.2f} | {t:+d}")

Class   |  x1    x2  | target
----------------------------
Class 1 | 0.50  2.50 | +1
Class 1 | 1.00  2.00 | +1
Class 1 | 1.00  3.00 | +1
Class 1 | 1.25  2.50 | +1
Class 1 | 1.50  2.00 | +1
Class 1 | 1.50  3.00 | +1
Class 2 | 0.00  2.00 | -1
Class 2 | 0.00  3.00 | -1
Class 2 | 1.00  1.00 | -1
Class 2 | 1.00  4.00 | -1
Class 2 | 2.00  2.00 | -1
Class 2 | 2.00  4.00 | -1


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Madaline vs Single-Layer Networks</h2>

| | Perceptron | Adaline | Madaline |
|---|---|---|---|
| **Layers** | 1 | 1 | 2 (hidden Adalines + fixed output unit) |
| **Learning signal** | thresholded output | linear net input (MSE) | thresholded outputs (MR-I) |
| **Trained weights** | all | all | hidden units only |
| **Decision region** | one half-plane | one half-plane | intersection of half-planes (AND), or its complement (OR) |
| **Convergence** | guaranteed for separable data | converges to the minimum MSE | **not guaranteed** |

**Strengths of Madaline**

- It separates classes with a **polygon-shaped** region, such as a triangle or a box, not only with a line.
- It solves problems that a perceptron or an Adaline **cannot** solve, such as **XOR** or **one class surrounded by another**.

**Weaknesses of Madaline**

- The **number of hidden units must be chosen by hand**. A region with k sides needs at least k units.
- The output unit is **fixed**, and the hidden outputs are **hard ±1 values**, so no gradient reaches the hidden layer. MR-I is a **heuristic** and may get stuck.
- With a single AND (or OR) output the region is **convex** (or the complement of a convex region). A non-convex class needs more layers or other logic.

The next cell checks the first claim: how well can **any single line** separate this data?

</div>

In [2]:
# Best accuracy that ANY single straight line can reach on DATA_B
import math

def best_single_line_accuracy(data, directions=720):
    best = 0.0

    for s in range(directions):
        angle = math.pi * s / directions
        c, d = math.cos(angle), math.sin(angle)

        # Project the points on the line direction and sort them
        proj = sorted((c*x + d*y, t) for (x, y), t in data)

        # A line is a cut between two neighbouring projections
        for cut in range(len(proj) + 1):
            if 0 < cut < len(proj) and proj[cut][0] - proj[cut-1][0] < 1e-9:
                continue                                    # equal projections cannot be cut

            left, right = proj[:cut], proj[cut:]
            plus_on_right = sum(t == -1 for _, t in left) + sum(t == 1 for _, t in right)
            plus_on_left = len(data) - plus_on_right
            best = max(best, plus_on_right / len(data), plus_on_left / len(data))

    return best

if __name__ == "__main__":
    acc = best_single_line_accuracy(DATA_B)
    print(f"Best accuracy of a single line: {acc:.1%}  ({round(acc * len(DATA_B))} of {len(DATA_B)} points)")

Best accuracy of a single line: 75.0%  (9 of 12 points)


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Network Design</h2>

The orange class has to be **enclosed**, so we need a region with enough sides:

- A **triangle** has **3 sides**, so **3 hidden units** are enough: one line on the left, one above and one below.
- The output unit is an **AND** of the three hidden outputs: **y = sign( z₁ + z₂ + z₃ − 2 )**. It gives **+1** only when all three units say +1, that is, **inside the triangle**.

Before training, we can check this design with lines chosen **by hand**:

- **Left line:** x₁ − 0.25 = 0
- **Upper line:** −1.15x₁ − x₂ + 5 = 0
- **Lower line:** −0.8x₁ + x₂ − 0.75 = 0

</div>

In [3]:
# Madaline: hidden Adaline units (trained) + fixed AND output unit
def sign(v):
    return 1 if v >= 0 else -1

def hidden_nets(x, units):
    return [w1*x[0] + w2*x[1] + b for w1, w2, b in units]

def predict(x, units):
    z = [sign(net) for net in hidden_nets(x, units)]
    return sign(sum(z) - (len(units) - 1))          # fixed AND output unit

def accuracy(data, units):
    return sum(predict(x, units) == t for x, t in data) / len(data)

# A hand-made solution with 3 hidden units: (w1, w2, b)
HAND_MADE = [(1.0, 0.0, -0.25),      # left line
             (-1.15, -1.0, 5.0),     # upper line
             (-0.8, 1.0, -0.75)]     # lower line

if __name__ == "__main__":
    print("x1    x2  |  z1  z2  z3 |  y | target")
    print("-------------------------------------")

    for x, t in DATA_B:
        z = [sign(net) for net in hidden_nets(x, HAND_MADE)]
        print(f"{x[0]:4.2f}  {x[1]:4.2f} | {z[0]:3} {z[1]:3} {z[2]:3} | {predict(x, HAND_MADE):2} | {t:4}")

    print(f"\nAccuracy of the hand-made network = {accuracy(DATA_B, HAND_MADE):.0%}")

x1    x2  |  z1  z2  z3 |  y | target
-------------------------------------
0.50  2.50 |   1   1   1 |  1 |    1
1.00  2.00 |   1   1   1 |  1 |    1
1.00  3.00 |   1   1   1 |  1 |    1
1.25  2.50 |   1   1   1 |  1 |    1
1.50  2.00 |   1   1   1 |  1 |    1
1.50  3.00 |   1   1   1 |  1 |    1
0.00  2.00 |  -1   1   1 | -1 |   -1
0.00  3.00 |  -1   1   1 | -1 |   -1
1.00  1.00 |   1   1  -1 | -1 |   -1
1.00  4.00 |   1  -1   1 | -1 |   -1
2.00  2.00 |   1   1  -1 | -1 |   -1
2.00  4.00 |   1  -1   1 | -1 |   -1

Accuracy of the hand-made network = 100%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Madaline Rule I (MR-I)</h2>

The network must now **find the lines by itself**. For every training point **(x, t)**, in random order:

1. Compute the hidden outputs **zₖ = sign(netₖ)** and the AND output **y**.
2. If **y = t**, nothing changes.
3. If **y ≠ t**, choose the units to correct:
   - **t = +1:** the output is wrong because some units say −1, so **all units with zₖ = −1** are corrected.
   - **t = −1:** all units say +1, so it is enough to flip **one** of them: the unit with the **net closest to 0** (the cheapest change).
4. Correct the chosen units with the **delta rule** toward the target:  
   **wₖ ← wₖ + α · (t − netₖ) · x**,  **bₖ ← bₖ + α · (t − netₖ)**

Training stops after the first epoch **without any error**. MR-I does not look for a **margin**, so a learned line may pass very close to a point.

**Initial lines:** every hidden unit starts with a random direction, and its line passes **near the centre of the data**. Lines that start far away from the data would never touch it.

</div>

In [4]:
# Madaline training with MR-I (pure Python)
import random

def init_units(data, n_units, rng):
    cx = sum(x[0] for x, _ in data) / len(data)
    cy = sum(x[1] for x, _ in data) / len(data)
    units = []

    for _ in range(n_units):
        w1, w2 = rng.uniform(-1, 1), rng.uniform(-1, 1)
        b = -(w1*cx + w2*cy) + rng.uniform(-0.3, 0.3)     # the line passes near the centre
        units.append([w1, w2, b])

    return units

def train_madaline(data, n_units=3, alpha=0.05, max_epochs=2000, seed=0):

    rng = random.Random(seed)
    units = init_units(data, n_units, rng)
    order = list(range(len(data)))
    history = []                                      # number of errors in every epoch

    for epoch in range(max_epochs):
        rng.shuffle(order)
        errors = 0

        for i in order:
            x, t = data[i]
            nets = hidden_nets(x, units)
            z = [sign(net) for net in nets]

            if sign(sum(z) - (n_units - 1)) == t:
                continue
            errors += 1

            if t == 1:
                to_fix = [k for k in range(n_units) if z[k] == -1]               # all that say -1
            else:
                to_fix = [min(range(n_units), key=lambda k: abs(nets[k]))]        # closest to 0

            for k in to_fix:
                error = t - nets[k]
                units[k][0] += alpha * error * x[0]
                units[k][1] += alpha * error * x[1]
                units[k][2] += alpha * error

        history.append(errors)
        if errors == 0:
            break

    return units, history

def converged(history):
    return history[-1] == 0

# Per-class accuracy (recall)
def recalls(data, units):
    r = []
    for target in (1, -1):
        pts = [x for x, t in data if t == target]
        r.append(sum(predict(x, units) == target for x in pts) / len(pts))
    return r

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Training the Madaline</h2>

The experiment uses these parameters:

- **3** hidden units
- Learning rate **α = 0.05**
- At most **2000** epochs
- Random seed **1**

<div align="center">
    <img src="images/lines_b.png" width="600">
    <br>
    <em><b>The three learned lines enclose the orange class (shaded region)</b></em>
</div>

</div>

In [5]:
# Train a Madaline with 3 hidden units on DATA_B
if __name__ == "__main__":
    units_b, history_b = train_madaline(DATA_B, n_units=3, alpha=0.05, max_epochs=2000, seed=1)

    print("Errors per epoch:\n")
    print("Epoch | Errors")
    print("--------------")
    for epoch, errors in enumerate(history_b, start=1):
        print(f"{epoch:5} | {errors:6}")

    print(f"\nConverged = {converged(history_b)}, epochs = {len(history_b)}")

    print("\nLearned lines (net = 0):\n")
    print("Unit | Line")
    print("---------------------------------------")
    for k, (w1, w2, b) in enumerate(units_b, start=1):
        print(f"  {k}  | {w1:+.3f} x1 {w2:+.3f} x2 {b:+.3f} = 0")

    r1, r2 = recalls(DATA_B, units_b)
    print(f"\nAccuracy = {accuracy(DATA_B, units_b):.0%}  (Class 1: {r1:.0%}, Class 2: {r2:.0%})")

Errors per epoch:

Epoch | Errors
--------------
    1 |      6
    2 |      6
    3 |      6
    4 |      5
    5 |      5
    6 |      2
    7 |      3
    8 |      2
    9 |      2
   10 |      4
   11 |      0

Converged = True, epochs = 11

Learned lines (net = 0):

Unit | Line
---------------------------------------
  1  | -0.580 x1 +0.940 x2 -0.750 = 0
  2  | -0.117 x1 -0.214 x2 +0.926 = 0
  3  | +0.711 x1 +0.639 x2 -1.917 = 0

Accuracy = 100%  (Class 1: 100%, Class 2: 100%)


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">How Reliable Is the Training?</h2>

MR-I has **no convergence guarantee**, so we repeat the training from **100 different random starts** and change the number of hidden units:

- **1 unit** is a single line, and **2 units** give only a wedge. Neither can enclose the class.
- **3 units** can form the triangle, but the training may get stuck.
- **More units** give MR-I more freedom, so it succeeds more often.

</div>

In [6]:
# Success rate for different numbers of hidden units (100 random starts each)
if __name__ == "__main__":
    print("Hidden units | converged runs | median epochs")
    print("--------------------------------------------")

    for n_units in [1, 2, 3, 4, 5]:
        epochs = []
        for seed in range(100):
            _, history = train_madaline(DATA_B, n_units=n_units, alpha=0.05, max_epochs=1000, seed=seed)
            if converged(history):
                epochs.append(len(history))

        median = sorted(epochs)[len(epochs) // 2] if epochs else None
        print(f"{n_units:12} | {len(epochs):9} / 100 | {median if median else '-':>13}")

Hidden units | converged runs | median epochs
--------------------------------------------
           1 |         0 / 100 |             -
           2 |         0 / 100 |             -
           3 |        92 / 100 |            21
           4 |        99 / 100 |            15
           5 |       100 / 100 |            10


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">More Data: 94 New Points</h2>

Now **94 random points** are added to the middle of the plane:

- **x₁** ~ Gaussian(mean **1**, std **0.2**)
- **x₂** ~ Gaussian(mean **2.5**, std **0.2**)

The new points are class 1, together with the 6 orange points. The data now has **100 points in class 1** (red) and only **6 points in class 2** (blue).

<div align="center">
    <img src="images/data_c.png" width="550">
</div>

</div>

In [7]:
# Add 94 Gaussian points to class 1
random.seed(42)                                   # fixed seed, so every run gives the same data
N_NEW = 94

new_points = [(random.gauss(1, 0.2), random.gauss(2.5, 0.2)) for _ in range(N_NEW)]
DATA_C = [(p, 1) for p in CLASS1 + new_points] + [(p, -1) for p in CLASS2]

def mean(values):
    return sum(values) / len(values)

def std(values):
    m = mean(values)
    return (sum((v - m) ** 2 for v in values) / len(values)) ** 0.5

if __name__ == "__main__":
    n1 = sum(t == 1 for _, t in DATA_C)
    n2 = sum(t == -1 for _, t in DATA_C)
    print(f"Class 1: {n1} points | Class 2: {n2} points\n")

    xs = [p[0] for p in new_points]
    ys = [p[1] for p in new_points]
    print(f"New points: mean x1 = {mean(xs):.3f}, std x1 = {std(xs):.3f}")
    print(f"            mean x2 = {mean(ys):.3f}, std x2 = {std(ys):.3f}")

Class 1: 100 points | Class 2: 6 points

New points: mean x1 = 1.032, std x1 = 0.183
            mean x2 = 2.510, std x2 = 0.181


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Training on the Larger Data Set</h2>

The same network (3 hidden units, α = 0.05) is trained on all **106 points**.

<div align="center">
    <img src="images/lines_c.png" width="600">
    <br>
    <em><b>Separating lines for the unbalanced data (100 vs 6 points)</b></em>
</div>

</div>

In [8]:
# Train on DATA_C
if __name__ == "__main__":
    units_c, history_c = train_madaline(DATA_C, n_units=3, alpha=0.05, max_epochs=2000, seed=1)

    print("Epoch | Errors")
    print("--------------")
    for epoch, errors in enumerate(history_c, start=1):
        print(f"{epoch:5} | {errors:6}")

    print(f"\nConverged = {converged(history_c)}, epochs = {len(history_c)}")

    print("\nLearned lines (net = 0):\n")
    for k, (w1, w2, b) in enumerate(units_c, start=1):
        print(f"Unit {k} | {w1:+.3f} x1 {w2:+.3f} x2 {b:+.3f} = 0")

    r1, r2 = recalls(DATA_C, units_c)
    print(f"\nAccuracy = {accuracy(DATA_C, units_c):.1%}  (Class 1: {r1:.0%}, Class 2: {r2:.0%})")

Epoch | Errors
--------------
    1 |     11
    2 |      6
    3 |      7
    4 |      6
    5 |      4
    6 |      4
    7 |      4
    8 |      2
    9 |      2
   10 |      2
   11 |      0

Converged = True, epochs = 11

Learned lines (net = 0):

Unit 1 | -0.616 x1 +0.849 x2 -0.757 = 0
Unit 2 | +0.066 x1 -0.280 x2 +0.829 = 0
Unit 3 | +0.700 x1 +0.754 x2 -1.862 = 0

Accuracy = 100.0%  (Class 1: 100%, Class 2: 100%)


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Does Class Imbalance Matter?</h2>

To see whether the **unequal number of points** changes the **separability**, the **number of epochs** or the quality of the result, we keep the 6 blue points and grow class 1 from **6** to **500** points.  
Every case is trained **20 times** from different random starts (at most 500 epochs).

Two details matter when the classes are unbalanced:

- **Accuracy is misleading.** A network that always answers "class 1" is already right on **N₁ / (N₁ + N₂)** of the points. So we report the **accuracy of every class** (recall) as well.
- A failed run can still show a very high total accuracy while it **misclassifies the few points of the small class**.

<div align="center">
    <img src="images/imbalance.png" width="900">
</div>

</div>

In [9]:
# Grow class 1 and keep the 6 blue points
def make_dataset(n_new, seed=7):
    rng = random.Random(seed)
    new = [(rng.gauss(1, 0.2), rng.gauss(2.5, 0.2)) for _ in range(n_new)]
    return [(p, 1) for p in CLASS1 + new] + [(p, -1) for p in CLASS2]

imbalance_results = []     # one row per dataset size

if __name__ == "__main__":
    print("  N1 |  N2 | baseline | converged | median epochs | accuracy | recall C1 | recall C2")
    print("---------------------------------------------------------------------------------")

    for n_new in [0, 10, 30, 94, 200, 500]:
        data = make_dataset(n_new)
        n1 = sum(t == 1 for _, t in data)
        n2 = len(data) - n1

        epochs, accs, rec1, rec2 = [], [], [], []
        for seed in range(20):
            units, history = train_madaline(data, n_units=3, alpha=0.05, max_epochs=500, seed=seed)
            if converged(history):
                epochs.append(len(history))
            r1, r2 = recalls(data, units)
            accs.append(accuracy(data, units)); rec1.append(r1); rec2.append(r2)

        median = sorted(epochs)[len(epochs) // 2] if epochs else None
        imbalance_results.append({"n1": n1, "n2": n2, "converged": len(epochs), "median": median,
                                  "acc": mean(accs), "rec1": mean(rec1), "rec2": mean(rec2)})
        print(f"{n1:4} | {n2:3} | {n1 / len(data):8.1%} | {len(epochs):6} /20 | {median:13} | "
              f"{mean(accs):8.1%} | {mean(rec1):9.1%} | {mean(rec2):9.1%}")

  N1 |  N2 | baseline | converged | median epochs | accuracy | recall C1 | recall C2
---------------------------------------------------------------------------------
   6 |   6 |    50.0% |     20 /20 |            19 |   100.0% |    100.0% |    100.0%
  16 |   6 |    72.7% |     20 /20 |            16 |   100.0% |    100.0% |    100.0%
  36 |   6 |    85.7% |     19 /20 |            18 |    98.6% |     98.5% |     99.2%
 100 |   6 |    94.3% |     20 /20 |            12 |   100.0% |    100.0% |    100.0%
 206 |   6 |    97.2% |     19 /20 |            12 |   100.0% |    100.0% |     98.3%
 506 |   6 |    98.8% |     20 /20 |            15 |   100.0% |    100.0% |    100.0%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

**What the experiment shows:**

- **Separability does not change.** The new points are very close together (std 0.2) and far from the blue points, so the same triangle still encloses them.
- **The number of epochs stays about the same.** It does not grow with N₁. What changes is the **cost of one epoch**, which is proportional to the number of points.
- **The failed runs come from the random start, not from the imbalance.** At most 1 run out of 20 gets stuck for every N₁, and the number does not grow with N₁.
- **Total accuracy hides the problem.** The baseline of always answering "class 1" is already above 90% for N₁ ≥ 100, and the small class is the one that suffers when a run gets stuck (see the recall of class 2 in the rows with 19 / 20).

The imbalance would matter more if the large class were **spread out** and overlapped the small one, because then the many class-1 points would dominate the updates.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>Conclusion</h2>
In this notebook, we built a **Madaline** with only pure Python and used it to separate a class that is **surrounded** by another class.

Through our experiments, we demonstrated:
- **Beyond a Single Line:** No single line classifies more than **9 of the 12 points (75%)**, while a Madaline with **3 hidden Adalines** and a fixed **AND** output separates all of them by enclosing the inner class in a triangle. It needed only **11 epochs**.
- **A Heuristic, Not a Guarantee:** MR-I corrects only the units that can fix an error, but it may get stuck. With 3 units, 92 out of 100 random starts converge, with 5 units all of them do, and with 1 or 2 units none does.
- **Class Imbalance:** Growing class 1 from 6 to 500 points did not change the separability or the number of epochs for this compact data. The 100 vs 6 case of the assignment was separated perfectly. It made plain accuracy misleading, so **per-class accuracy** should always be checked.

Madaline shows why **hidden layers** matter, and it leads naturally to multi-layer networks trained with **backpropagation**.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>👩‍💻 About the Author</h2>

<b>Sarah M Afshar</b>  
<i>Python Developer | Generative AI Specialist</i>

I am a passionate developer focused on the intersection of <b>Large Language Models (LLMs)</b>, neural network architectures, and the evolution of intelligent systems.  
This project is part of my journey in exploring the mathematical foundations of AI.

🔗 GitHub: https://github.com/afshars  

🔗 LinkedIn: https://www.linkedin.com/in/sara-m-afshar/

</div>